索引不是数据本身，而是一份额外维护的“查找目录”
1. 查找user_id为xx的数据
2. 有没有合适的索引？
3. 有-> 查索引定位候选记录 -> 读取对应表数据(index Scan)
4. 没有 -> 扫描大量表记录（seq scan）

### 为什么不把所有字段都建立索引？
索引越多，插入越慢，更新越慢，删除维护越多，更占空间。

索引是用额外空间和写入成本换取部分查询更快。

索引利读，不利写；按查询建，不按字段建。

### 主键为什么不再建索引？

因为已经会为主键创建唯一索引，所以主键索引是默认索引。

### EXPLAIN 是什么？
```sql
EXPLAIN
SELECT *
FROM users
WHERE email = 'alice@example.com';
```
展示计划，不真正执行查询。

#### EXPLAIN ANALYZE
不仅展示计划，还真证执行sql，并记录实际耗时和行数。

EXPLAIN 看计划，ANALYZE 真执行。

### 复合索引
`(user_id, status, created_at)`
复合索引是把多个字段组合在一起，复合 B-Tree 索引在查询限制前面的字段时通常最有效。

复合索引像多层目录，查询最好从第一层开始。

### 三种特殊索引
- 表达式索引：先加工再建立目录
- 部分索引：只给部分数据建立目录
- 覆盖索引：把查询需要的附加数据也放进目录

### 表达式索引
查询对字段做固定加工，目录也保存加工后的结果。

```
普通索引目录：
Alice@Example.com → 位置

实际查询：
alice@example.com
```


### 覆盖索引
```sql
CREATE INDEX idx_orders_user
ON orders (user_id)
INCLUDE (status, amount);
```
索引中不仅保存 user_id 的值，还保存 status 和 amount 的值。
> 覆盖索引让目录不只告诉你在哪，还顺便带回需要的信息。


### 部分索引
假设订单有 1000 万条，990万条历史或者已经删除。接口几乎只查询未删除。
```sql
CREATE INDEX idx_active_orders
ON orders (user_id)
WHERE deleted_at IS NULL;
```
索引中只包含未删除订单

> 部分索引不是给所有数据建目录，只收录经常查询的那部分。

### Seq Scan
`Seq Scan on orders` 顺序检查表中的数据
### index Scan
`Index Scan using idx_orders_user_id`先通过索引定位，再去表中读取完整记录
### Index Only Scan
`Index Only Scan using idx_orders_user_cover`查询需要的信息可能已经包含在索引中。可以少做或避免普通意义上的回表读取

### Bitmap Index Scan 和 Bitmap Heap Scan
分为2步：
1. Bitmap Index Scan -> 从索引中收集一批记录位置
2. Bitmap Heap Scan -> 按更适合批量读取的方式访问表

可以理解成：
- 需要取1件商品：查目录后直接取
- 需要取100万件：直接扫仓库
- 需要取中等数量：先圈出一批货架，再批量取


极少数据    → Index Scan

中等数量    → Bitmap Scan

大部分数据  → Seq Scan